# Дообучение TNC-DBGC* (mDeBERTa-v3 + BiGRU + CRF) на LLMTrace_detection

Самодостаточный ноутбук для Kaggle (GPU T4, 16 GB). Дообучает потокенный теггер ИИ-текста
(архитектура `DeBERTaBiGRUCRFTagger` из `src/model.py`) на датасете
[`iitolstykh/LLMTrace_detection`](https://huggingface.co/datasets/iitolstykh/LLMTrace_detection) (Apache-2.0), языки `ru` и `eng`.

**Протокол оценки**
* модель (эпоха) и порог документа выбираются **только на validation**; test используется один раз в конце;
* score документа = средняя маргинальная вероятность класса AI по словам (forward-backward CRF), альтернатива — доля слов, помеченных AI декодером Витерби;
* порог: максимум F1 (позитив = ai + mixed) при FPR на human <= 0.20 (иначе — максимум F1);
* на test: ROC-AUC (human vs ai+mixed) с bootstrap 95% ДИ, P/R/F1 и FPR при откалиброванном пороге, ROC-AUC mixed vs human,
  символьная F1 локализации, пословная точность; всё — также по языкам.

**Выходные файлы** (`/kaggle/working`): `tnc_dbgc_llmtrace.pt`, `calibration.json`, `metrics.json`,
`training_params.json`, `roc_curve_llmtrace.png`.

Перед запуском: включите *Accelerator → GPU T4* и *Internet → On* (см. `kaggle/README.md`).

In [ ]:
# ==== Установка зависимостей и проверка GPU ====
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("pytorch-crf", "torchcrf")
_ensure("sentencepiece")
_ensure("datasets")
_ensure("protobuf", "google.protobuf")

import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory / 2**30:.1f} GB, sm_{p.major}{p.minor}")
else:
    print("!!! GPU не найден. Settings -> Accelerator -> GPU T4 x2 (или P100). На CPU обучение займёт много часов.")

In [ ]:
# ==== КОНФИГУРАЦИЯ (меняйте здесь) ====
# Инициализация весов:
#   "checkpoint" — с вашего чекпойнта tnc_dbgc_coat_best.pt (загружен как Kaggle Dataset);
#   "base"       — с нуля от microsoft/mdeberta-v3-base (голова BiGRU+CRF инициализируется случайно).
INIT_FROM = "checkpoint"
CHECKPOINT_PATH = "/kaggle/input/tnc-dbgc-coat/tnc_dbgc_coat_best.pt"  # если не найден — ищем *.pt в /kaggle/input
BASE_MODEL_NAME = "microsoft/mdeberta-v3-base"
HIDDEN_DIM = 256          # для INIT_FROM="checkpoint" берётся из чекпойнта

# Данные
DATASET_NAME = "iitolstykh/LLMTrace_detection"
DATASET_REVISION = None   # None = ветка main; фактический commit sha пишется в training_params.json
LANGS = ["ru", "eng"]
MAX_TRAIN_SAMPLES = 20000  # из 54 311 (None = все)
MAX_VAL_SAMPLES = 3000     # из 11 443 (None = все)
MAX_TEST_SAMPLES = None    # из 13 588 (None = все; 5000 — быстрее)
SEED = 42

# Обучение
EPOCHS = 2
MAX_LEN = 512
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
GRAD_ACCUM = 1
# Послойные learning rate (LLRD, как в src/optim.py): эмбеддинги / нижние слои / верхние слои / BiGRU+CRF
LR_EMB, LR_LOW, LR_HIGH, LR_HEAD = 1e-6, 5e-6, 1e-5, 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
FP16 = True                       # torch.cuda.amp (autocast + GradScaler)
LABEL_ALL_SUBTOKENS = True        # True — как в src/dataset.py (все субтокены слова получают метку слова);
                                  # False — только первый субтокен, остальные -100 (для CRF они станут 0)
TRAIN_MAX_CHUNKS_PER_DOC = 2      # длинные тексты режутся на окна <= MAX_LEN токенов; сколько окон брать в train
EVAL_MAX_CHUNKS_PER_DOC = 4       # сколько окон оценивать (слова за пределами не учитываются в метриках)
SAVE_EVERY_STEPS = 1000           # промежуточное сохранение tnc_dbgc_llmtrace_last.pt
NUM_WORKERS = 2

# Оценка
DOC_SCORE = "marginal"            # "marginal" (средняя P(AI) по словам) | "viterbi_frac" (доля AI-слов по Витерби)
MAX_FPR = 0.20                    # ограничение FPR на human при выборе порога (только validation)
BOOTSTRAP_ITERS = 1000
EVAL_BASELINE = False             # True — дополнительно оценить исходную модель до дообучения (+~15-25 мин)

OUTPUT_DIR = "/kaggle/working"
FINAL_CKPT_NAME = "tnc_dbgc_llmtrace.pt"
KEEP_LAST_CKPT = False            # удалить промежуточный *_last.pt в конце (экономия места в Output)

In [ ]:
# ==== Импорты, сид, устройство ====
import os, json, time, glob, random, platform
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from datasets import load_dataset
from torchcrf import CRF

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(FP16 and DEVICE.type == "cuda")
OUT = Path(OUTPUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
FINAL_CKPT = OUT / FINAL_CKPT_NAME
LAST_CKPT = OUT / FINAL_CKPT_NAME.replace(".pt", "_last.pt")
T_START = time.time()
print("device:", DEVICE, "| amp:", USE_AMP, "| output:", OUT)

In [ ]:
# ==== Чистые вспомогательные функции (без torch): слова, метки, чанки, метрики ====
import re
import math
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, roc_auc_score, roc_curve,
)

WORD_RE = re.compile(r"\S+")  # то же разбиение, что text.split() в src/dataset.py и src/pipeline.py


def word_spans(text):
    # Список (start, end) символьных границ слов (последовательностей непробельных символов).
    return [(m.start(), m.end()) for m in WORD_RE.finditer(text)]


def normalize_intervals(intervals, text_len):
    out = []
    for iv in intervals or []:
        if iv is None or len(iv) < 2:
            continue
        s, e = max(0, int(iv[0])), min(text_len, int(iv[1]))
        if e > s:
            out.append((s, e))
    return sorted(out)


def ai_char_counts(text_len, spans, intervals):
    # Для каждого слова: сколько его символов попадает в объединение AI-интервалов.
    if not spans:
        return np.zeros(0, dtype=np.int64)
    mask = np.zeros(text_len, dtype=np.int64)
    for s, e in intervals:
        mask[s:e] = 1
    cs = np.concatenate([[0], np.cumsum(mask)])
    starts = np.array([s for s, _ in spans], dtype=np.int64)
    ends = np.array([e for _, e in spans], dtype=np.int64)
    return cs[ends] - cs[starts]


def word_labels_from_intervals(text, spans, intervals):
    # Метка слова = 1, если строго больше половины его символов лежит в AI-интервалах.
    counts = ai_char_counts(len(text), spans, intervals)
    lens = np.array([e - s for s, e in spans], dtype=np.int64)
    labels = (counts * 2 > lens).astype(np.int64)
    return labels, counts, lens


def chunk_items(word_ids, n_words, budget, max_chunks=None):
    # word_ids: результат encoding.word_ids() для токенизации БЕЗ спецтокенов (каждый токен -> слово).
    # Жадно режет слова на чанки так, чтобы в чанке было <= budget субтокенов.
    # Возвращает список чанков: dict(w_start, w_end, tok_start, tok_end, first_pos={word: pos_in_seq},
    # tok_words=[word для каждого токена чанка]); pos_in_seq учитывает [CLS] в позиции 0.
    counts = [0] * n_words
    for w in word_ids:
        if w is not None:
            counts[w] += 1
    prefix = [0]
    for c in counts:
        prefix.append(prefix[-1] + c)
    bounds = []
    ws, cur = 0, 0
    for w in range(n_words):
        c = counts[w]
        if cur > 0 and cur + c > budget:
            bounds.append((ws, w))
            ws, cur = w, 0
        cur += c
    if ws < n_words:
        bounds.append((ws, n_words))
    if max_chunks:
        bounds = bounds[:max_chunks]
    chunks = []
    for ws, we in bounds:
        t0 = prefix[ws]
        t1 = min(prefix[we], t0 + budget)  # одно гигантское слово обрезается
        tok_words = [word_ids[t] for t in range(t0, t1)]
        first_pos = {}
        for i, w in enumerate(tok_words):
            if w not in first_pos:
                first_pos[w] = i + 1  # +1 за [CLS]
        chunks.append(dict(w_start=ws, w_end=we, tok_start=t0, tok_end=t1,
                           first_pos=first_pos, tok_words=tok_words))
    return chunks


def binary_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    fpr = fp / (fp + tn) if fp + tn else 0.0
    acc = (tp + tn) / max(len(y_true), 1)
    return dict(precision=prec, recall=rec, f1=f1, fpr=fpr, accuracy=acc, tp=tp, fp=fp, tn=tn, fn=fn)


def select_threshold(scores, y_true, max_fpr=0.20):
    # Порог на ВАЛИДАЦИИ: максимум F1 (позитив = ai+mixed) при FPR на human <= max_fpr.
    # Если ни один порог не удовлетворяет ограничению — просто максимум F1 (fallback).
    # Предсказание: score >= threshold.
    scores = np.asarray(scores, dtype=float)
    y = np.asarray(y_true).astype(int)
    cands = np.unique(scores)
    cands = np.concatenate([cands, [cands[-1] + 1e-6]]) if len(cands) else np.array([0.5])
    best, best_any = None, None
    for t in cands:
        m = binary_metrics(y, scores >= t)
        key = (m["f1"], -m["fpr"])
        if best_any is None or key > best_any[0]:
            best_any = (key, float(t), m)
        if m["fpr"] <= max_fpr and (best is None or key > best[0]):
            best = (key, float(t), m)
    fallback = best is None
    _, thr, m = best_any if fallback else best
    return dict(threshold=thr, fallback_used=fallback, max_fpr=max_fpr,
                **{f"val_{k}": v for k, v in m.items()})


def safe_auc(y, s):
    y = np.asarray(y).astype(int)
    if len(np.unique(y)) < 2:
        return float("nan")
    return float(roc_auc_score(y, s))


def bootstrap_auc_ci(y, s, iters=1000, seed=42, alpha=0.05):
    y = np.asarray(y).astype(int)
    s = np.asarray(s, dtype=float)
    if len(np.unique(y)) < 2:
        return dict(low=float("nan"), high=float("nan"), n_valid=0)
    rng = np.random.RandomState(seed)
    n = len(y)
    vals = []
    for _ in range(iters):
        idx = rng.randint(0, n, n)
        yy = y[idx]
        if yy.min() == yy.max():
            continue
        vals.append(roc_auc_score(yy, s[idx]))
    vals = np.array(vals)
    return dict(low=float(np.quantile(vals, alpha / 2)), high=float(np.quantile(vals, 1 - alpha / 2)),
                std=float(vals.std()), n_valid=int(len(vals)))


# ---- Метрики последовательностей (копия src/metrics.py) ----
def extract_boundaries(seq):
    return [i for i in range(1, len(seq)) if seq[i] != seq[i - 1]]


def compute_boundary_mae(true_seqs, pred_seqs):
    errors = []
    for t, p in zip(true_seqs, pred_seqs):
        tb = next((i for i in range(1, len(t)) if t[i] != t[i - 1]), len(t))
        pb = next((i for i in range(1, len(p)) if p[i] != p[i - 1]), len(p))
        errors.append(abs(tb - pb))
    return float(np.mean(errors)) if errors else 0.0


def compute_f1_at_k(true_seqs, pred_seqs, k=3, tolerance=2):
    scores = []
    for t, p in zip(true_seqs, pred_seqs):
        gt, pr = extract_boundaries(t), extract_boundaries(p)[:k]
        if not gt and not pr:
            scores.append(1.0); continue
        if not gt or not pr:
            scores.append(0.0); continue
        matched, used = 0, set()
        for pb in pr:
            for i, gb in enumerate(gt):
                if i not in used and abs(pb - gb) <= tolerance:
                    matched += 1; used.add(i); break
        scores.append(2.0 * matched / (len(pr) + len(gt)))
    return float(np.mean(scores)) if scores else 0.0


def evaluate_sequences(true_seqs, pred_seqs):
    ts = [np.asarray(s, dtype=np.int64) for s in true_seqs if len(s) > 0]
    ps = [np.asarray(s, dtype=np.int64) for s in pred_seqs if len(s) > 0]
    if not ts:
        return dict(accuracy=0.0, precision=0.0, recall=0.0, f1=0.0, mcc=0.0, kappa=0.0,
                    boundary_mae=0.0, f1_at_3=0.0)
    ft, fp = np.concatenate(ts), np.concatenate(ps)
    return dict(
        accuracy=float(accuracy_score(ft, fp)),
        precision=float(precision_score(ft, fp, average="weighted", zero_division=0)),
        recall=float(recall_score(ft, fp, average="weighted", zero_division=0)),
        f1=float(f1_score(ft, fp, average="weighted", zero_division=0)),
        mcc=float(matthews_corrcoef(ft, fp)),
        kappa=float(cohen_kappa_score(ft, fp)),
        boundary_mae=compute_boundary_mae(true_seqs, pred_seqs),
        f1_at_3=compute_f1_at_k(true_seqs, pred_seqs, k=3, tolerance=2),
    )


# ---- Документные и символьные метрики ----
# record = dict(label="human"|"ai"|"mixed", lang, score_marginal, score_viterbi,
#               word_pred, word_gold, ai_counts, word_lens)  (все массивы — по покрытым словам)
def char_counts(rec):
    pred = np.asarray(rec["word_pred"]).astype(bool)
    ai = np.asarray(rec["ai_counts"], dtype=np.int64)
    ln = np.asarray(rec["word_lens"], dtype=np.int64)
    tp = int(ai[pred].sum())
    fp = int((ln - ai)[pred].sum())
    fn = int(ai[~pred].sum())
    return tp, fp, fn


def char_f1(records):
    # Микро-F1 по символам: предсказанные AI-символы = символы слов, помеченных AI;
    # эталон = символы внутри ai_char_intervals. Считается по непробельным символам (символам слов).
    tp = fp = fn = 0
    for r in records:
        a, b, c = char_counts(r)
        tp += a; fp += b; fn += c
    p = tp / (tp + fp) if tp + fp else 0.0
    rc = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * rc / (p + rc) if p + rc else 0.0
    return dict(precision=p, recall=rc, f1=f, tp_chars=tp, fp_chars=fp, fn_chars=fn)


def doc_score(rec, score_type):
    return rec["score_marginal"] if score_type == "marginal" else rec["score_viterbi"]


def compute_metrics(records, threshold, score_type="marginal", bootstrap_iters=1000, seed=42):
    out = dict(n_docs=len(records))
    if not records:
        return out
    labels = np.array([r["label"] for r in records])
    out["n_by_label"] = {k: int((labels == k).sum()) for k in ("human", "ai", "mixed")}
    y = (labels != "human").astype(int)
    s = np.array([doc_score(r, score_type) for r in records], dtype=float)
    s_alt = np.array([doc_score(r, "viterbi_frac" if score_type == "marginal" else "marginal")
                      for r in records], dtype=float)
    out["score_type"] = score_type
    out["roc_auc"] = safe_auc(y, s)
    out["roc_auc_ci95"] = bootstrap_auc_ci(y, s, bootstrap_iters, seed)
    out["roc_auc_alt_score"] = safe_auc(y, s_alt)
    mh = labels != "ai"
    out["roc_auc_mixed_vs_human"] = safe_auc(y[mh], s[mh])
    out["roc_auc_mixed_vs_human_ci95"] = bootstrap_auc_ci(y[mh], s[mh], bootstrap_iters, seed)
    ah = labels != "mixed"
    out["roc_auc_ai_vs_human"] = safe_auc(y[ah], s[ah])
    m = binary_metrics(y, s >= threshold)
    out["threshold"] = float(threshold)
    out.update({f"doc_{k}": v for k, v in m.items()})
    out["fpr_human"] = m["fpr"]
    pred_pos = s >= threshold
    out["recall_by_label"] = {k: float(pred_pos[labels == k].mean()) if (labels == k).any() else float("nan")
                              for k in ("ai", "mixed")}
    out["char_localization_ai_mixed"] = char_f1([r for r in records if r["label"] != "human"])
    out["char_localization_mixed"] = char_f1([r for r in records if r["label"] == "mixed"])
    out["char_localization_all"] = char_f1(records)
    wg = np.concatenate([np.asarray(r["word_gold"]) for r in records]) if records else np.zeros(0)
    wp = np.concatenate([np.asarray(r["word_pred"]) for r in records]) if records else np.zeros(0)
    out["word_accuracy"] = float((wg == wp).mean()) if len(wg) else float("nan")
    out["word_level"] = evaluate_sequences([list(r["word_gold"]) for r in records],
                                           [list(r["word_pred"]) for r in records])
    return out


def compute_metrics_by_lang(records, threshold, score_type="marginal", bootstrap_iters=1000, seed=42):
    res = {"overall": compute_metrics(records, threshold, score_type, bootstrap_iters, seed), "by_lang": {}}
    for lang in sorted(set(r["lang"] for r in records)):
        sub = [r for r in records if r["lang"] == lang]
        res["by_lang"][lang] = compute_metrics(sub, threshold, score_type, bootstrap_iters, seed)
    return res


def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        f = float(o)
        return None if math.isnan(f) or math.isinf(f) else f
    if isinstance(o, np.ndarray):
        return to_jsonable(o.tolist())
    return o

print("helpers ok")

In [ ]:
# ==== Модель TNC-DBGC* (копия src/model.py; ключи state_dict совпадают) ====
# Отличия от src/model.py только во внутренней логике (не в параметрах):
#  * эмиссии приводятся к float32, а CRF считается вне autocast — устойчиво при fp16;
#  * добавлен метод emissions() и функция crf_marginals() (forward-backward) для документного score.


class DynamicDropout(nn.Module):
    # Dropout с вероятностью, плавно растущей от initial_p до max_p по ходу обучения.
    def __init__(self, initial_p: float = 0.1, max_p: float = 0.3):
        super().__init__()
        self.initial_p = initial_p
        self.max_p = max_p
        self.current_p = initial_p

    def set_progress(self, progress: float) -> None:
        clamped = min(max(progress, 0.0), 1.0)
        self.current_p = self.initial_p + (self.max_p - self.initial_p) * clamped

    def forward(self, x):
        return nn.functional.dropout(x, p=self.current_p, training=self.training)


def _load_transformer(model_name):
    try:
        return AutoModel.from_pretrained(model_name, dtype=torch.float32)
    except TypeError:
        return AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)


class DeBERTaBiGRUCRFTagger(nn.Module):
    def __init__(self, model_name="microsoft/mdeberta-v3-base", num_labels=2, hidden_dim=256,
                 num_gru_layers=2, dropout_initial=0.1, dropout_max=0.3):
        super().__init__()
        self.num_labels = num_labels
        self.transformer = _load_transformer(model_name)
        self.transformer.float()
        transformer_hidden = self.transformer.config.hidden_size

        self.dyn_dropout1 = DynamicDropout(dropout_initial, dropout_max)
        self.dyn_dropout2 = DynamicDropout(dropout_initial, dropout_max)

        self.gru = nn.GRU(
            input_size=transformer_hidden,
            hidden_size=hidden_dim,
            num_layers=num_gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout_max if num_gru_layers > 1 else 0.0,
        )
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self.proj = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.act = nn.ReLU()
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

        self.crf = CRF(num_labels, batch_first=True)

    def update_dropout(self, progress: float) -> None:
        self.dyn_dropout1.set_progress(progress)
        self.dyn_dropout2.set_progress(progress)

    def emissions(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        seq_out = self.dyn_dropout1(outputs.last_hidden_state.float())
        gru_out, _ = self.gru(seq_out)
        gru_out = self.layer_norm(gru_out)
        gru_out = self.dyn_dropout2(self.act(self.proj(gru_out)))
        return self.classifier(gru_out).float()  # [B, L, num_labels]

    def forward(self, input_ids, attention_mask, labels=None):
        emissions = self.emissions(input_ids, attention_mask)
        mask = attention_mask.bool()
        with torch.autocast(device_type=emissions.device.type, enabled=False):
            if labels is not None:
                crf_labels = labels.clone()
                crf_labels[crf_labels == -100] = 0
                return -self.crf(emissions, crf_labels, mask=mask, reduction="mean")
            return self.crf.decode(emissions, mask=mask)


def crf_marginals(crf, emissions, mask):
    # Маргинальные вероятности меток P(y_t = c | x) алгоритмом forward-backward.
    # emissions [B, L, C] float32, mask [B, L] bool (маска выровнена влево, mask[:, 0] = True).
    em = emissions.float()
    B, L, C = em.shape
    trans = crf.transitions.float()          # [C, C]: из i в j
    start, end = crf.start_transitions.float(), crf.end_transitions.float()
    alphas = [start.unsqueeze(0) + em[:, 0]]
    for t in range(1, L):
        nxt = torch.logsumexp(alphas[-1].unsqueeze(2) + trans.unsqueeze(0), dim=1) + em[:, t]
        alphas.append(torch.where(mask[:, t].unsqueeze(1), nxt, alphas[-1]))
    betas = [None] * L
    betas[L - 1] = end.unsqueeze(0).expand(B, C)
    for t in range(L - 2, -1, -1):
        cand = torch.logsumexp(trans.unsqueeze(0) + (em[:, t + 1] + betas[t + 1]).unsqueeze(1), dim=2)
        betas[t] = torch.where(mask[:, t + 1].unsqueeze(1), cand, end.unsqueeze(0).expand(B, C))
    alpha = torch.stack(alphas, 1)             # [B, L, C]
    beta = torch.stack(betas, 1)
    log_z = torch.logsumexp(alphas[-1] + end.unsqueeze(0), dim=1)  # [B]
    return torch.exp(alpha + beta - log_z.view(B, 1, 1))


# --- самопроверка forward-backward полным перебором на маленьком примере ---
def _check_marginals():
    import itertools
    torch.manual_seed(0)
    crf = CRF(2, batch_first=True)
    em = torch.randn(2, 5, 2)
    mask = torch.tensor([[1, 1, 1, 1, 1], [1, 1, 1, 0, 0]], dtype=torch.bool)
    marg = crf_marginals(crf, em, mask)
    for b in range(2):
        n = int(mask[b].sum())
        brute = torch.zeros(n, 2)
        logs, seqs = [], list(itertools.product([0, 1], repeat=n))
        for seq in seqs:
            s = crf.start_transitions[seq[0]] + em[b, 0, seq[0]] + crf.end_transitions[seq[-1]]
            for t in range(1, n):
                s = s + crf.transitions[seq[t - 1], seq[t]] + em[b, t, seq[t]]
            logs.append(s)
        p = torch.softmax(torch.stack(logs).detach(), 0)
        for seq, pp in zip(seqs, p):
            for t, c in enumerate(seq):
                brute[t, c] += pp
        assert torch.allclose(marg[b, :n].detach(), brute, atol=1e-5), (marg[b, :n], brute)
    print("crf_marginals: OK (совпадает с полным перебором)")

_check_marginals()

In [ ]:
# ==== Оптимизатор LLRD (как src/optim.py) и сохранение чекпойнтов (формат src/train.py) ====
def build_llrd_optimizer(model, weight_decay=0.01):
    layers = list(model.transformer.encoder.layer)
    mid = len(layers) // 2
    groups = [
        {"params": model.transformer.embeddings.parameters(), "lr": LR_EMB, "weight_decay": weight_decay},
        {"params": model.transformer.encoder.layer[:mid].parameters(), "lr": LR_LOW, "weight_decay": weight_decay},
        {"params": model.transformer.encoder.layer[mid:].parameters(), "lr": LR_HIGH, "weight_decay": weight_decay},
        {"params": model.gru.parameters(), "lr": LR_HEAD, "weight_decay": weight_decay},
        {"params": model.layer_norm.parameters(), "lr": LR_HEAD, "weight_decay": 0.0},
        {"params": model.proj.parameters(), "lr": LR_HEAD, "weight_decay": weight_decay},
        {"params": model.classifier.parameters(), "lr": LR_HEAD, "weight_decay": weight_decay},
        {"params": model.crf.parameters(), "lr": LR_HEAD, "weight_decay": 0.0},
    ]
    # Параметры энкодера вне embeddings/layer (например, rel_embeddings/LayerNorm DeBERTa) — в группу верхних слоёв
    groups = [dict(g, params=list(g["params"])) for g in groups]
    covered = {id(p) for g in groups for p in g["params"]}
    rest = [p for p in model.parameters() if id(p) not in covered]
    if rest:
        groups.append({"params": rest, "lr": LR_HIGH, "weight_decay": weight_decay})
        print(f"LLRD: {sum(p.numel() for p in rest):,} доп. параметров (rel_embeddings и т.п.) -> lr={LR_HIGH}")
    return torch.optim.AdamW(groups, lr=LR_HIGH, weight_decay=weight_decay)


def save_checkpoint(model, model_name, hidden_dim, metrics, path):
    tmp = str(path) + ".tmp"
    torch.save({
        "model_state_dict": model.state_dict(),
        "model_name": model_name,
        "hidden_dim": hidden_dim,
        "val_metrics": to_jsonable(metrics),
    }, tmp)
    os.replace(tmp, path)


def resolve_checkpoint_path(path):
    if path and os.path.isfile(path):
        return path
    found = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True)) + \
            sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True))
    if found:
        pref = [f for f in found if "tnc_dbgc" in os.path.basename(f)]
        chosen = (pref or found)[0]
        print(f"Чекпойнт {path!r} не найден, найден автоматически: {chosen}")
        if len(found) > 1:
            print("  все кандидаты:", found)
        return chosen
    raise FileNotFoundError(
        f"Чекпойнт {path!r} не найден и в /kaggle/input нет *.pt. Добавьте Dataset с файлом "
        "(Add Input) или поставьте INIT_FROM = 'base'.")

In [ ]:
# ==== Загрузка LLMTrace_detection и подвыборка ====
dataset_sha = None
try:
    from huggingface_hub import HfApi
    dataset_sha = HfApi().dataset_info(DATASET_NAME, revision=DATASET_REVISION).sha
except Exception as e:
    print("Не удалось получить sha датасета:", repr(e))
print(f"Датасет {DATASET_NAME} @ {DATASET_REVISION or 'main'} (sha={dataset_sha})")

raw = load_dataset(DATASET_NAME, revision=DATASET_REVISION)
print(raw)

def take(split, n):
    ds = raw[split]
    full_n = len(ds)
    ds = ds.filter(lambda b: [l in LANGS for l in b["lang"]], batched=True)
    ds = ds.shuffle(seed=SEED)  # датасет упорядочен по языку — перемешивание обязательно
    if n is not None and n < len(ds):
        ds = ds.select(range(n))
    rows = ds.to_list()
    print(f"{split}: {full_n} всего -> {len(rows)} выбрано;",
          dict(Counter((r['lang'], r['label']) for r in rows)))
    return rows, full_n

train_rows, n_train_full = take("train", MAX_TRAIN_SAMPLES)
val_rows, n_val_full = take("validation", MAX_VAL_SAMPLES)
test_rows, n_test_full = take("test", MAX_TEST_SAMPLES)
del raw

In [ ]:
# ==== Токенизация, пословная разметка, нарезка на окна ====
# Слова = непробельные последовательности (\S+) с символьными смещениями.
# Метка слова = 1, если > 50% его символов попадает в ai_char_intervals.
if INIT_FROM == "checkpoint":
    CKPT_IN = resolve_checkpoint_path(CHECKPOINT_PATH)
    _meta = torch.load(CKPT_IN, map_location="cpu", weights_only=False)
    MODEL_NAME = _meta.get("model_name", BASE_MODEL_NAME)
    HIDDEN_DIM = int(_meta.get("hidden_dim", HIDDEN_DIM))
    INIT_VAL_METRICS = _meta.get("val_metrics", {})
    INIT_STATE = _meta["model_state_dict"]
    del _meta
    print(f"Чекпойнт: {CKPT_IN} | model_name={MODEL_NAME} | hidden_dim={HIDDEN_DIM}")
else:
    CKPT_IN, MODEL_NAME, INIT_VAL_METRICS, INIT_STATE = None, BASE_MODEL_NAME, {}, None

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.is_fast, "Нужен fast-токенизатор (word_ids). Проверьте sentencepiece/protobuf."
CLS_ID, SEP_ID = tokenizer.cls_token_id, tokenizer.sep_token_id
PAD_ID = tokenizer.pad_token_id or 0
BUDGET = MAX_LEN - 2
ANOMALIES = Counter()


def prepare_doc(row, max_chunks):
    text = row["text"] or ""
    spans = word_spans(text)
    if not spans:
        ANOMALIES["empty_text"] += 1
        return None
    intervals = normalize_intervals(row.get("ai_char_intervals"), len(text))
    if row["label"] == "ai" and not intervals:
        ANOMALIES["ai_without_intervals->full"] += 1
        intervals = [(0, len(text))]
    if row["label"] == "human" and intervals:
        ANOMALIES["human_with_intervals"] += 1
    labels, ai_counts, lens = word_labels_from_intervals(text, spans, intervals)
    words = [text[s:e] for s, e in spans]
    enc = tokenizer(words, is_split_into_words=True, add_special_tokens=False, truncation=False)
    ids, wids = enc["input_ids"], enc.word_ids()
    chunks = chunk_items(wids, len(words), BUDGET, max_chunks)
    n_cov = chunks[-1]["w_end"] if chunks else 0
    items = []
    for ch in chunks:
        input_ids = [CLS_ID] + ids[ch["tok_start"]:ch["tok_end"]] + [SEP_ID]
        tok_labels = [-100] * len(input_ids)
        if LABEL_ALL_SUBTOKENS:
            for i, w in enumerate(ch["tok_words"]):
                tok_labels[i + 1] = int(labels[w])
        else:
            for w, pos in ch["first_pos"].items():
                tok_labels[pos] = int(labels[w])
        items.append(dict(input_ids=input_ids, labels=tok_labels,
                          positions=list(ch["first_pos"].items())))  # [(word_idx, pos_in_seq)]
    return dict(label=row["label"], lang=row["lang"], n_words=len(words), n_cov=n_cov,
                word_gold=labels[:n_cov], ai_counts=ai_counts[:n_cov], word_lens=lens[:n_cov],
                n_chars=len(text), items=items)


def prepare_split(rows, max_chunks, name):
    docs = []
    for r in tqdm(rows, desc=f"prep {name}"):
        d = prepare_doc(r, max_chunks)
        if d is not None:
            docs.append(d)
    n_items = sum(len(d["items"]) for d in docs)
    cov = sum(d["n_cov"] for d in docs) / max(sum(d["n_words"] for d in docs), 1)
    print(f"{name}: docs={len(docs)}, окон={n_items}, покрытие слов={cov:.3f}")
    return docs

train_docs = prepare_split(train_rows, TRAIN_MAX_CHUNKS_PER_DOC, "train")
val_docs = prepare_split(val_rows, EVAL_MAX_CHUNKS_PER_DOC, "val")
test_docs = prepare_split(test_rows, EVAL_MAX_CHUNKS_PER_DOC, "test")
print("Аномалии разметки:", dict(ANOMALIES))

train_items = [it for d in train_docs for it in d["items"]]
_tl = np.array([l for it in train_items for l in it["labels"] if l != -100])
print(f"train окон: {len(train_items)}, доля AI-токенов: {_tl.mean():.3f}, "
      f"средняя длина: {np.mean([len(it['input_ids']) for it in train_items]):.0f} токенов")
del train_rows, val_rows, test_rows, _tl


class ItemDataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]


def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        att[i, :n] = 1
        lab[i, :n] = torch.tensor(b["labels"])
    return {"input_ids": ids, "attention_mask": att, "labels": lab}


def make_batches(lengths, bs, seed):
    # Батчи из окон похожей длины (меньше паддинга), порядок батчей случайный.
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(lengths))
    mega = bs * 50
    batches = []
    for i in range(0, len(idx), mega):
        part = sorted(idx[i:i + mega].tolist(), key=lambda j: lengths[j])
        batches += [part[k:k + bs] for k in range(0, len(part), bs)]
    rng.shuffle(batches)
    return batches

In [ ]:
# ==== Создание модели и инициализация весов ====
model = DeBERTaBiGRUCRFTagger(model_name=MODEL_NAME, num_labels=2, hidden_dim=HIDDEN_DIM)
if INIT_STATE is not None:
    res = model.load_state_dict(INIT_STATE, strict=True)
    print("Веса загружены из чекпойнта:", res)
    print("val_metrics исходного чекпойнта:", INIT_VAL_METRICS)
    del INIT_STATE
else:
    print(f"Старт с {MODEL_NAME} (BiGRU/CRF — случайная инициализация)")
model.to(DEVICE)
print(f"Параметров: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

In [ ]:
# ==== Инференс по документам: Витерби + маргиналы -> пословные предсказания ====
@torch.no_grad()
def predict_docs(model, docs, desc="predict"):
    model.eval()
    flat = [(di, it) for di, d in enumerate(docs) for it in d["items"]]
    flat.sort(key=lambda x: len(x[1]["input_ids"]))
    preds = [np.zeros(d["n_cov"], dtype=np.int64) for d in docs]
    margs = [np.zeros(d["n_cov"], dtype=np.float32) for d in docs]
    for k in tqdm(range(0, len(flat), EVAL_BATCH_SIZE), desc=desc, leave=False):
        chunk = flat[k:k + EVAL_BATCH_SIZE]
        batch = collate([it for _, it in chunk])
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        att = batch["attention_mask"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            em = model.emissions(ids, att)
        em = em.float()
        mask = att.bool()
        vit = model.crf.decode(em, mask=mask)
        p_ai = crf_marginals(model.crf, em, mask)[..., 1].cpu().numpy()
        for row, (di, it) in enumerate(chunk):
            for w, pos in it["positions"]:
                preds[di][w] = vit[row][pos]
                margs[di][w] = p_ai[row, pos]
    records = []
    for d, p, m in zip(docs, preds, margs):
        records.append(dict(
            label=d["label"], lang=d["lang"],
            score_marginal=float(m.mean()) if len(m) else 0.0,
            score_viterbi=float(p.mean()) if len(p) else 0.0,
            word_pred=p, word_marg=m, word_gold=d["word_gold"],
            ai_counts=d["ai_counts"], word_lens=d["word_lens"],
        ))
    return records


def quick_val_summary(records):
    # Метрики для выбора эпохи (только validation): ROC-AUC документа + пословные метрики (как src/metrics.py).
    y = np.array([r["label"] != "human" for r in records]).astype(int)
    s = np.array([doc_score(r, DOC_SCORE) for r in records])
    labels = np.array([r["label"] for r in records])
    mh = labels != "ai"
    summ = evaluate_sequences([list(r["word_gold"]) for r in records], [list(r["word_pred"]) for r in records])
    summ.update(doc_roc_auc=safe_auc(y, s), doc_roc_auc_mixed_vs_human=safe_auc(y[mh], s[mh]),
                char_f1_ai_mixed=char_f1([r for r in records if r["label"] != "human"])["f1"])
    return summ

In [ ]:
# ==== (опционально) Оценка исходной модели до дообучения ====
BASELINE = None
if EVAL_BASELINE:
    t0 = time.time()
    _vr = predict_docs(model, val_docs, "baseline val")
    _cal = select_threshold([doc_score(r, DOC_SCORE) for r in _vr],
                            [int(r["label"] != "human") for r in _vr], MAX_FPR)
    _tr = predict_docs(model, test_docs, "baseline test")
    BASELINE = dict(calibration=_cal,
                    test=compute_metrics_by_lang(_tr, _cal["threshold"], DOC_SCORE, BOOTSTRAP_ITERS, SEED))
    o = BASELINE["test"]["overall"]
    print(f"BASELINE test: ROC-AUC={o['roc_auc']:.4f}  F1={o['doc_f1']:.4f}  FPR={o['fpr_human']:.4f}  "
          f"charF1(ai+mixed)={o['char_localization_ai_mixed']['f1']:.4f}  ({time.time() - t0:.0f} c)")
    del _vr, _tr
else:
    print("EVAL_BASELINE=False — пропускаем")

In [ ]:
# ==== Обучение ====
optimizer = build_llrd_optimizer(model, WEIGHT_DECAY)
train_lengths = [len(it["input_ids"]) for it in train_items]
n_batches = len(make_batches(train_lengths, BATCH_SIZE, SEED))
steps_per_epoch = math.ceil(n_batches / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print(f"Батчей/эпоху: {n_batches}, шагов оптимизатора всего: {total_steps}")
history, best_auc, best_epoch, global_step, skipped = [], -1.0, None, 0, 0
train_ds = ItemDataset(train_items)
T_TRAIN = time.time()

for epoch in range(EPOCHS):
    model.train()
    batches = make_batches(train_lengths, BATCH_SIZE, SEED + epoch)
    loader = DataLoader(train_ds, batch_sampler=batches, collate_fn=collate,
                        num_workers=NUM_WORKERS, pin_memory=True)
    run_loss, n_loss = 0.0, 0
    pbar = tqdm(enumerate(loader), total=len(batches), desc=f"Epoch {epoch + 1}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)
    for step, batch in pbar:
        model.update_dropout((epoch + step / len(batches)) / EPOCHS)
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        att = batch["attention_mask"].to(DEVICE, non_blocking=True)
        lab = batch["labels"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            loss = model(ids, att, labels=lab)
        if not torch.isfinite(loss):
            skipped += 1
            optimizer.zero_grad(set_to_none=True)
            continue
        scaler.scale(loss / GRAD_ACCUM).backward()
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(batches):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            if global_step % SAVE_EVERY_STEPS == 0:
                save_checkpoint(model, MODEL_NAME, HIDDEN_DIM, {}, LAST_CKPT)
        run_loss += loss.item(); n_loss += 1
        if step % 20 == 0:
            pbar.set_postfix(loss=f"{run_loss / n_loss:.3f}", drop=f"{model.dyn_dropout1.current_p:.2f}",
                             lr=f"{scheduler.get_last_lr()[-1]:.1e}")
    save_checkpoint(model, MODEL_NAME, HIDDEN_DIM, {}, LAST_CKPT)

    val_records = predict_docs(model, val_docs, f"val ep{epoch + 1}")
    vm = quick_val_summary(val_records)
    vm.update(epoch=epoch + 1, train_loss=run_loss / max(n_loss, 1), skipped_nonfinite=skipped,
              elapsed_min=(time.time() - T_TRAIN) / 60)
    history.append(vm)
    print(f"\nЭпоха {epoch + 1}: loss={vm['train_loss']:.4f} | val doc ROC-AUC={vm['doc_roc_auc']:.4f} | "
          f"mixed-vs-human={vm['doc_roc_auc_mixed_vs_human']:.4f} | word F1={vm['f1']:.4f} | "
          f"kappa={vm['kappa']:.4f} | charF1={vm['char_f1_ai_mixed']:.4f} | {vm['elapsed_min']:.1f} мин")
    _auc = vm["doc_roc_auc"] if np.isfinite(vm["doc_roc_auc"]) else -1.0
    if best_epoch is None or _auc > best_auc:
        best_auc, best_epoch = _auc, epoch + 1
        save_checkpoint(model, MODEL_NAME, HIDDEN_DIM, vm, FINAL_CKPT)
        print(f"  --> лучший чекпойнт сохранён: {FINAL_CKPT}")

TRAIN_MINUTES = (time.time() - T_TRAIN) / 60
print(f"Обучение заняло {TRAIN_MINUTES:.1f} мин; лучшая эпоха {best_epoch} (val ROC-AUC {best_auc:.4f}); "
      f"пропущено нечисловых батчей: {skipped}")

In [ ]:
# ==== Загрузка лучшей модели и калибровка порога ТОЛЬКО на validation ====
_ck = torch.load(FINAL_CKPT, map_location="cpu", weights_only=False)
model.load_state_dict(_ck["model_state_dict"])
best_val_metrics = _ck["val_metrics"]
del _ck
model.to(DEVICE)

val_records = predict_docs(model, val_docs, "val (best)")
calibration = select_threshold([doc_score(r, DOC_SCORE) for r in val_records],
                               [int(r["label"] != "human") for r in val_records], MAX_FPR)
calibration.update(score_type=DOC_SCORE, score_definition=(
    "mean P(AI) over words (CRF forward-backward marginal at first subtoken)" if DOC_SCORE == "marginal"
    else "fraction of words tagged AI by CRF Viterbi"),
    positive_class="ai+mixed", rule=f"max F1 s.t. FPR(human) <= {MAX_FPR}; fallback max F1",
    word_split=r"regex \S+", max_len=MAX_LEN, eval_max_chunks_per_doc=EVAL_MAX_CHUNKS_PER_DOC,
    n_val_docs=len(val_records), best_epoch=best_epoch, model_name=MODEL_NAME,
    checkpoint=FINAL_CKPT_NAME, dataset=DATASET_NAME, dataset_sha=dataset_sha)
val_metrics_full = compute_metrics_by_lang(val_records, calibration["threshold"], DOC_SCORE, BOOTSTRAP_ITERS, SEED)
with open(OUT / "calibration.json", "w") as f:
    json.dump(to_jsonable(calibration), f, indent=2, ensure_ascii=False)
print(json.dumps(to_jsonable(calibration), indent=2, ensure_ascii=False))

# В чекпойнт кладём полные метрики validation + порог (формат ключей как в src/train.py)
best_val_metrics = dict(best_val_metrics, threshold=calibration["threshold"], score_type=DOC_SCORE,
                        val_doc_f1=calibration["val_f1"], val_doc_fpr=calibration["val_fpr"])
save_checkpoint(model, MODEL_NAME, HIDDEN_DIM, best_val_metrics, FINAL_CKPT)

In [ ]:
# ==== Финальная оценка на test (порог зафиксирован на validation) ====
test_records = predict_docs(model, test_docs, "test")
THR = calibration["threshold"]
test_metrics = compute_metrics_by_lang(test_records, THR, DOC_SCORE, BOOTSTRAP_ITERS, SEED)

def _fmt(name, m):
    ci = m["roc_auc_ci95"]
    print(f"[{name}] n={m['n_docs']} {m['n_by_label']}\n"
          f"  ROC-AUC={m['roc_auc']:.4f} (95% CI {ci['low']:.4f}-{ci['high']:.4f}) | "
          f"ROC-AUC mixed vs human={m['roc_auc_mixed_vs_human']:.4f} | ai vs human={m['roc_auc_ai_vs_human']:.4f}\n"
          f"  @thr={THR:.4f}: P={m['doc_precision']:.4f} R={m['doc_recall']:.4f} F1={m['doc_f1']:.4f} "
          f"FPR(human)={m['fpr_human']:.4f} | recall ai={m['recall_by_label']['ai']:.3f} "
          f"mixed={m['recall_by_label']['mixed']:.3f}\n"
          f"  char F1: ai+mixed={m['char_localization_ai_mixed']['f1']:.4f} "
          f"mixed={m['char_localization_mixed']['f1']:.4f} all={m['char_localization_all']['f1']:.4f} | "
          f"word acc={m['word_accuracy']:.4f} | word kappa={m['word_level']['kappa']:.4f}")

_fmt("TEST overall", test_metrics["overall"])
for lang, m in test_metrics["by_lang"].items():
    _fmt(f"TEST {lang}", m)

In [ ]:
# ==== ROC-кривые (test) ====
import matplotlib.pyplot as plt

def _roc(records, keep=lambda r: True):
    rs = [r for r in records if keep(r)]
    y = np.array([r["label"] != "human" for r in rs]).astype(int)
    s = np.array([doc_score(r, DOC_SCORE) for r in rs])
    if len(np.unique(y)) < 2:
        return None
    fpr, tpr, _ = roc_curve(y, s)
    return fpr, tpr, roc_auc_score(y, s)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
curves_left = [("все (human vs ai+mixed)", lambda r: True)] + \
              [(f"{l}", (lambda l: lambda r: r["lang"] == l)(l)) for l in sorted(test_metrics["by_lang"])]
curves_right = [("mixed vs human", lambda r: r["label"] != "ai"), ("ai vs human", lambda r: r["label"] != "mixed")]
for ax, curves, title in [(axes[0], curves_left, "Test ROC: human vs ai+mixed"),
                          (axes[1], curves_right, "Test ROC по типу документа")]:
    for name, keep in curves:
        res = _roc(test_records, keep)
        if res:
            ax.plot(res[0], res[1], lw=2, label=f"{name}: AUC={res[2]:.3f}")
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax.axvline(MAX_FPR, color="red", ls=":", lw=1, label=f"FPR={MAX_FPR}")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(title); ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "roc_curve_llmtrace.png", dpi=150)
plt.show()

In [ ]:
# ==== Сохранение metrics.json и training_params.json ====
def _counts(docs):
    return {"docs": len(docs), "windows": sum(len(d["items"]) for d in docs),
            "by_lang_label": {f"{k[0]}/{k[1]}": v for k, v in Counter((d["lang"], d["label"]) for d in docs).items()}}

metrics = {
    "model": FINAL_CKPT_NAME, "dataset": DATASET_NAME, "dataset_sha": dataset_sha,
    "protocol": "epoch selection and threshold on validation only; test evaluated once",
    "calibration": calibration,
    "test": test_metrics,
    "validation": val_metrics_full,
    "training_history": history,
    "baseline_before_finetune": BASELINE,
}
with open(OUT / "metrics.json", "w") as f:
    json.dump(to_jsonable(metrics), f, indent=2, ensure_ascii=False)

config = {k: v for k, v in globals().items()
          if k.isupper() and isinstance(v, (int, float, str, bool, list, type(None))) and not k.startswith("_")}
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
params = {
    "config": config,
    "dataset": {"name": DATASET_NAME, "revision": DATASET_REVISION, "sha": dataset_sha, "license": "Apache-2.0",
                "langs": LANGS, "split_sizes_full": {"train": n_train_full, "validation": n_val_full, "test": n_test_full},
                "used": {"train": _counts(train_docs), "validation": _counts(val_docs), "test": _counts(test_docs)},
                "label_anomalies": dict(ANOMALIES),
                "word_label_rule": "word=AI if >50% of its chars fall inside ai_char_intervals; words = regex \\S+"},
    "init": {"from": INIT_FROM, "checkpoint": CKPT_IN, "model_name": MODEL_NAME, "hidden_dim": HIDDEN_DIM,
             "checkpoint_val_metrics": INIT_VAL_METRICS},
    "training": {"epochs": EPOCHS, "steps": global_step, "batches_per_epoch": n_batches, "best_epoch": best_epoch,
                 "best_val_roc_auc": best_auc, "train_minutes": TRAIN_MINUTES, "skipped_nonfinite": skipped},
    "seed": SEED,
    "env": {"gpu": gpu, "torch": torch.__version__, "transformers": transformers.__version__,
            "datasets": datasets.__version__, "python": platform.python_version()},
    "total_minutes": (time.time() - T_START) / 60,
}
with open(OUT / "training_params.json", "w") as f:
    json.dump(to_jsonable(params), f, indent=2, ensure_ascii=False)

if not KEEP_LAST_CKPT and LAST_CKPT.exists():
    LAST_CKPT.unlink()
print(f"Готово за {(time.time() - T_START) / 60:.1f} мин. Файлы в {OUT}:")
for p in sorted(OUT.iterdir()):
    if p.is_file():
        print(f"  {p.name:32s} {p.stat().st_size / 2**20:10.1f} MB")

## Что скачать (вкладка *Output* справа или страница версии ноутбука → *Output*)

| Файл на Kaggle | Куда положить в репозитории |
|---|---|
| `tnc_dbgc_llmtrace.pt` | `models/tnc_dbgc_llmtrace.pt` |
| `calibration.json` | `models/lm_calibration.json` |
| `metrics.json` | `results/lm_metrics.json` |
| `training_params.json` | `results/lm_training_params.json` (по желанию) |
| `roc_curve_llmtrace.png` | `results/lm_roc_curve.png` (по желанию) |

Чекпойнт имеет тот же формат, что и `src/train.py` (`model_state_dict`, `model_name`, `hidden_dim`, `val_metrics`),
и загружается `EmailAnalysisPipeline(checkpoint_path="models/tnc_dbgc_llmtrace.pt")`.